Just a dev notebook to develop the algorithm. Will export the finals into a py file for execution. 

In [1]:
import os 
import sys

# set root folder to project root
root_path = os.path.abspath(os.path.join(".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# auto reload
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from pprint import pprint
from modules.bin_calcs import (
    PosturalAngles,
    create_rotation_matrices, 
    _compute_incremental_rotation_matrices, 
    _decompose_rotation_matrices_yxy, 
    _get_postural_angles, 
    _normalize_postural_angles,
    _extract_bin_data,
    _accumulate_euler_components,
    calculate_bin_rotations
)
from schema import Heatmap
from modules.data_loading import load_participant_details, load_motion_capture_data

participant_details = load_participant_details('../data/raw_normalized_data/participant_details.xlsx')
participant = participant_details[0]
data = load_motion_capture_data(participant.filename)

In [3]:
# create R matrices
rotation_matrices = create_rotation_matrices(data, "left")

In [6]:
# determine postural position
postural_angles = _get_postural_angles(rotation_matrices)
postural_angles_normalized = _normalize_postural_angles(postural_angles)

In [7]:
# compute relative motion between each frame
relative_matrices = _compute_incremental_rotation_matrices(rotation_matrices)

In [8]:
# decompose relative motion matrices into euler angles
euler_angles = _decompose_rotation_matrices_yxy(relative_matrices)

c:\Users\chris\anaconda3\envs\rtsa_mocap\Lib\site-packages\IPython\core\interactiveshell.py:3748: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
# calculate cumulative motion in each axis
side_object = getattr(participant, "left")  # or "right" depending on the side
initial_data_object: Heatmap = side_object.humerothoracic.heatmap

# for each bin
for elevation_range_start in range(0, 180, int(initial_data_object.bin_width)):

    for poe_range_start in range(0, 360, int(initial_data_object.bin_width)):
        # extract bin boundaries
        elevation_start = elevation_range_start
        elevation_end = elevation_range_start + initial_data_object.bin_width
        poe_start = poe_range_start
        poe_end = poe_range_start + initial_data_object.bin_width

        # filter extract only data within the bin boundaries
        bin_data = _extract_bin_data(
            mocap_data=relative_matrices, 
            postural_data=postural_angles_normalized, 
            elevation_start=elevation_start, 
            elevation_end=elevation_end, 
            poe_start=poe_start, 
            poe_end=poe_end
        )

        # return zero values if no data in the bin
        if bin_data.shape[0] == 0:
            initial_data_object.elevation = np.append(initial_data_object.elevation, 0)
            initial_data_object.poe = np.append(initial_data_object.poe, 0)
            initial_data_object.ir_er = np.append(initial_data_object.ir_er, 0)
            initial_data_object.cumulative_motion = np.append(initial_data_object.cumulative_motion, 0)
            initial_data_object.sample_count = np.append(initial_data_object.sample_count, 0)
            continue
        # else, decompose the bin data into euler angles
        else:
            euler_angles = _decompose_rotation_matrices_yxy(bin_data)

            # Calculate totals
            total_elevation, total_poe, total_irer = _accumulate_euler_components(euler_angles)
            total_motion = total_elevation + total_poe + total_irer
            n_samples = bin_data.shape[0]
        
            # save the bin calcs to the data object
            initial_data_object.elevation = np.append(initial_data_object.elevation, total_elevation)
            initial_data_object.poe = np.append(initial_data_object.poe, total_poe)
            initial_data_object.ir_er = np.append(initial_data_object.ir_er, total_irer)
            initial_data_object.cumulative_motion = np.append(initial_data_object.cumulative_motion, total_motion)
            initial_data_object.sample_count = np.append(initial_data_object.sample_count, n_samples)

In [ ]:
calc = calculate_bin_rotations(data, "left")

c:\Users\chris\Documents\School\THESIS\Publications\Third Submission\New Analysis\modules\bin_calcs.py:542: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  result = _calculate_single_bin(


In [11]:
print(initial_data_object == calc)

True
